### **Prerequisites for the Lab**

Before the session, ensure you have the environment set up.

1. Install [Ollama](https://ollama.com/) and download a lightweight model: `ollama run llama3` (or `llama3.1` or `mistral`).
2. Install the required Python libraries:

In [42]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.5 MB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,093 kB]
Fetched 13.6 MB in 2s (6,836 kB/s)
Reading package lists... Done

In [43]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [44]:
import os, subprocess, time

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Open a log file to catch the output so it doesn't fill the OS pipe
log_file = open('/content/ollama_server.log', 'w')

subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file
)

time.sleep(5)

In [45]:
!ollama pull gemma4:latest

In [46]:
!ollama pull nomic-embed-text

In [47]:
!pip install langchain langchain-ollama langchain-community chromadb langchain-chroma

In [48]:
import requests
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate

In [49]:
# 1. Initialize the local, secure model
llm = ChatOllama(model="gemma4:latest", temperature=0)

In [50]:
# 2. Define Procedural Memory (SOPs)
PROCEDURAL_RULES = {
    "weather_sop": "You must ALWAYS report the temperature. Always append 'Have a great day!' at the end."
}

In [51]:
print("1 Executed: Model loaded and Procedural Rules established.")
print(f"Loaded SOP: {PROCEDURAL_RULES['weather_sop']}")

1 Executed: Model loaded and Procedural Rules established.
Loaded SOP: You must ALWAYS report the temperature. Always append 'Have a great day!' at the end.


### **2: Episodic Memory (Long-Term Context)**

 This represents a backend database (like a CRM) that stores past interactions and known facts about a user. This prevents the agent from asking repetitive questions.

In [52]:
import chromadb
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

In [53]:
# 1. Initialize persistent database
client = chromadb.PersistentClient(path="./my_long_term_memory")
embedding_function = OllamaEmbeddings(model="nomic-embed-text")

In [54]:
# 2. Setup Vector Store
vector_db = Chroma(
    collection_name="user_context",
    embedding_function=embedding_function,
    persist_directory="./my_long_term_memory"
)

In [55]:
# 3. Add facts (The 'Embed' step)
vector_db.add_texts(
    texts=[
        "Usha is a corporate trainer based in Chennai, India. Prefers concise, professional answers.",
        "Alex is a software engineer based in London. Prefers technical details."
    ],
    metadatas=[{"username": "usha"}, {"username": "alex"}]
)

['9de357e5-803c-441b-b2e4-1e821090673f',
 'eeb40e80-4510-42f2-a63b-d1dd37bd8b4f']

In [56]:
# 4. Define the new, smart retrieval function
def retrieve_episodic_context(query: str) -> str:
    """Finds relevant long-term memory using semantic search."""
    # This searches by meaning, not just by key
    results = vector_db.similarity_search(query, k=1)
    return results[0].page_content if results else "No memory found."

### **3: Tool Calling (Live Open-Source API)**

 Show how the LLM reaches out to the real world. We test the tool in isolation here so can see exactly what JSON data the LLM will eventually parse.


In [57]:
import requests
from langchain_core.tools import tool

@tool
def get_live_weather(city: str) -> str:
    """Fetches real-time weather data for a given city using live geocoding and Open-Meteo API."""

    # 1. Resolve city name to coordinates using Open-Meteo Geocoding API
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1&language=en&format=json"
    geo_response = requests.get(geo_url).json()

    if not geo_response.get("results"):
        return f"Could not find coordinates for {city}."

    lat = geo_response["results"][0]["latitude"]
    lon = geo_response["results"][0]["longitude"]

    # 2. Fetch weather using resolved coordinates
    weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m"
    weather_response = requests.get(weather_url).json()

    temp = weather_response["current"]["temperature_2m"]
    unit = weather_response["current_units"]["temperature_2m"]

    return f"The current temperature in {city} is {temp}{unit}."

In [58]:
# Test the Tool directly
print("3 Executed: Testing Tool Calling independently.")
tool_result = get_live_weather.invoke({"city": "Chennai"})
print(f"Raw API Result: {tool_result}")

3 Executed: Testing Tool Calling independently.
Raw API Result: The current temperature in Chennai is 30.3°C.


### **4: Working & Short-Term Memory (The Agent Engine)**

 Now we introduce the Agent. **Working Memory** is the scratchpad where the agent thinks (`Thought -> Action -> Observation`). **Short-Term Memory** is the sliding window of the current chat history.


In [59]:
# Initialize Short-Term Memory (remembers the last 2 interactions)
short_term_memory = ConversationBufferWindowMemory(
    k=10,
    memory_key="chat_history",
    return_messages=True # <-- This is the magic parameter!
)

In [60]:
# Initialize the Agent with the Tool and Memory
agent = initialize_agent(
    tools=[get_live_weather],
    llm=llm,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    memory=short_term_memory,
    verbose=True # This visualizes the 'Working Memory' (the reasoning scratchpad)
)

In [61]:
print("4 Executed: Running the Agent Loop.")

4 Executed: Running the Agent Loop.


In [62]:
# Instead of f-string referencing a dictionary variable, call the function
# This dynamically fetches the best match based on the user's intent
context_found = retrieve_episodic_context("Where is Usha based?")

enriched_prompt = f"""
Context about the user: {context_found}
User Request: What is the weather like in my home city right now?
"""

In [63]:
# Run the agent
raw_agent_draft = agent.run(enriched_prompt)
print(f"\n[Agent Draft Output]: {raw_agent_draft}")



> Entering new AgentExecutor chain...
```json
{
    "action": "get_live_weather",
    "action_input": "Chennai, India"
}
```
Observation: The current temperature in Chennai, India is 30.3°C.
Thought:```json
{
    "action": "Final Answer",
    "action_input": "The current weather in Chennai is 30.3°C."
}
```

> Finished chain.

[Agent Draft Output]: The current weather in Chennai is 30.3°C.


In [64]:
# 2. Ask a follow-up question that relies on context
# Notice how the Agent doesn't need to ask "Which city?" again
raw_agent_draft = agent.run("Is it a good day to go for a walk there?")
print(f"\n[Agent Draft Output]: {raw_agent_draft}")



> Entering new AgentExecutor chain...
```json
{
    "action": "get_live_weather",
    "action_input": "Chennai"
}
```
Observation: The current temperature in Chennai is 30.0°C.
Thought:```json
{
    "action": "Final Answer",
    "action_input": "Given that the current temperature is 30.0°C, it will be quite warm for a walk. For maximum comfort, I recommend going early in the morning or late in the evening when the heat is less intense."
}
```

> Finished chain.

[Agent Draft Output]: Given that the current temperature is 30.0°C, it will be quite warm for a walk. For maximum comfort, I recommend going early in the morning or late in the evening when the heat is less intense.


In [65]:
#Contextual Refinement
# The agent remembers 'Chennai' was the location from the previous turn
raw_agent_draft = agent.run("What is the humidity like there?")
print(f"\n[Agent Draft Output]: {raw_agent_draft}")



> Entering new AgentExecutor chain...
```json
{
    "action": "get_live_weather",
    "action_input": "Chennai"
}
```
Observation: The current temperature in Chennai is 30.0°C.
Thought:```json
{
    "action": "Final Answer",
    "action_input": "The current temperature in Chennai is 30.0°C. While I do not have the specific humidity reading right now, generally, with this level of warmth, you should expect high humidity levels."
}
```

> Finished chain.

[Agent Draft Output]: The current temperature in Chennai is 30.0°C. While I do not have the specific humidity reading right now, generally, with this level of warmth, you should expect high humidity levels.


In [66]:
raw_agent_draft = agent.run("What about the wind speed?")
print(f"\n[Agent Draft Output]: {raw_agent_draft}")



> Entering new AgentExecutor chain...
```json
{
    "action": "get_live_weather",
    "action_input": "Chennai"
}
```
Observation: The current temperature in Chennai is 30.0°C.
Thought:```json
{
    "action": "Final Answer",
    "action_input": "The current temperature in Chennai is 30.0°C. Since the wind speed data was not available, please note that it remains quite warm, and going early morning or late evening would still be recommended for a comfortable walk."
}
```

> Finished chain.

[Agent Draft Output]: The current temperature in Chennai is 30.0°C. Since the wind speed data was not available, please note that it remains quite warm, and going early morning or late evening would still be recommended for a comfortable walk.


In [67]:
# This forces the agent to look back at the history to compare
raw_agent_draft = agent.run("If I travel to London, would the weather be better for a walk than here?")
print(f"\n[Agent Draft Output]: {raw_agent_draft}")



> Entering new AgentExecutor chain...
```json
{
    "action": "get_live_weather",
    "action_input": "London"
}
```
Observation: The current temperature in London is 24.5°C.
Thought:```json
{
    "action": "Final Answer",
    "action_input": "Based on the current weather in London (24.5°C), it would likely be more comfortable for a walk compared to Chennai's current warm conditions."
}
```

> Finished chain.

[Agent Draft Output]: Based on the current weather in London (24.5°C), it would likely be more comfortable for a walk compared to Chennai's current warm conditions.


In [68]:
print(short_term_memory.load_memory_variables({}))

{'chat_history': [HumanMessage(content='\nContext about the user: Usha is a corporate trainer based in Chennai, India. Prefers concise, professional answers.\nUser Request: What is the weather like in my home city right now?\n', additional_kwargs={}, response_metadata={}), AIMessage(content='The current weather in Chennai is 30.3°C.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Is it a good day to go for a walk there?', additional_kwargs={}, response_metadata={}), AIMessage(content='Given that the current temperature is 30.0°C, it will be quite warm for a walk. For maximum comfort, I recommend going early in the morning or late in the evening when the heat is less intense.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the humidity like there?', additional_kwargs={}, response_metadata={}), AIMessage(content='The current temperature in Chennai is 30.0°C. While

### **5: The Feedback Loop (Reflection & Compliance)**

 Enterprise AI must be safe. The agent above likely fetched the weather, but it might have forgotten to add the required "Have a great day!" phrase from our Procedural Memory. The Evaluator enforces the rules.


In [69]:
eval_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a strict Enterprise QA System.
    Review the draft below and ensure it strictly follows this SOP Rule: "{rule}"

    If it follows the rule perfectly, output the original draft.
    If it misses the rule, rewrite the draft to include it."""),
    ("user", "Draft to review:\n{draft}")
])

In [70]:
evaluator_chain = eval_prompt | llm

In [71]:
print("5 Executed: Running the Feedback/Evaluator Loop.")

5 Executed: Running the Feedback/Evaluator Loop.


In [72]:
# Run it through the evaluator using the rule we defined in above
final_corrected_output = evaluator_chain.invoke({
    "rule": PROCEDURAL_RULES["weather_sop"],
    "draft": raw_agent_draft
}).content

In [73]:
print(f"\n[FINAL QA APPROVED OUTPUT]:\n{final_corrected_output}")


[FINAL QA APPROVED OUTPUT]:
Based on the current weather in London (24.5°C), it would likely be more comfortable for a walk compared to Chennai's current warm conditions (approximately 31°C). Have a great day!
